In [36]:
pip install haversine

Note: you may need to restart the kernel to use updated packages.


In [37]:
pip install plotly

Note: you may need to restart the kernel to use updated packages.


In [38]:
%pip install -U nbformat

Note: you may need to restart the kernel to use updated packages.


In [39]:
import pandas as pd
import json
from haversine import haversine, Unit
import plotly.express as px

In [40]:
# load dataset
df_porto = pd.read_csv("porto/porto.csv")

In [41]:
# avoid changing the original df
df = df_porto.copy(deep = True)
display(df) 


,TRIP_ID,CALL_TYPE,ORIGIN_CALL,ORIGIN_STAND,TAXI_ID,TIMESTAMP,DAY_TYPE,MISSING_DATA,POLYLINE
0,1372636858620000589,C,NaN,NaN,20000589,1372636858,A,False,"[[-8.618643,41.141412],[-8.618499,41.141376],[..."
1,1372637303620000596,B,NaN,7.0,20000596,1372637303,A,False,"[[-8.639847,41.159826],[-8.640351,41.159871],[..."
2,1372636951620000320,C,NaN,NaN,20000320,1372636951,A,False,"[[-8.612964,41.140359],[-8.613378,41.14035],[-..."
3,1372636854620000520,C,NaN,NaN,20000520,1372636854,A,False,"[[-8.574678,41.151951],[-8.574705,41.151942],[..."
4,1372637091620000337,C,NaN,NaN,20000337,1372637091,A,False,"[[-8.645994,41.18049],[-8.645949,41.180517],[-..."
...,...,...,...,...,...,...,...,...,...
1710665,1404171463620000698,C,NaN,NaN,20000698,1404171463,A,False,"[[-8.612469,41.14602],[-8.612487,41.145993],[-..."
1710666,1404171367620000670,C,NaN,NaN,20000670,1404171367,A,False,"[[-8.610138,41.140845],[-8.610174,41.140935],[..."
1710667,1388745716620000264,C,NaN,NaN,20000264,1388745716,A,False,[]
1710668,1404141826620000248,B,NaN,12.0,20000248,1404141826,A,False,"[[-8.630712,41.154885],[-8.63073,41.154813],[-..."


## Part 1 - EDA
### Examine main attributes, get an overview of the data distribution and investigate the presence and impact of missing trajectories

#### TRIP_ID inspection

In [42]:
# check whether TRIP_ID is unique for each row
print("Number of unique TRIP_IDs:", len(df["TRIP_ID"].unique()))
print("Number of rows in the dataset:", len(df["TRIP_ID"]))
print("Number of duplicated TRIP_IDs:",  len(df["TRIP_ID"]) - len(df["TRIP_ID"].unique()))

# check how many rows are duplicated for each TRIP_ID and inspect trips with the most rows (value_counts sorts by TRIP_ID)
print("")
print("Number of duplicated rows for each TRIP_ID and inspection of the trip with the highest count:")
duplicate_ids_df = df[df["TRIP_ID"].duplicated(keep=False)].sort_values(by=["TRIP_ID"])
display(duplicate_ids_df.value_counts("TRIP_ID"))
display(duplicate_ids_df[duplicate_ids_df["TRIP_ID"] == 1397172149620000454 ])

# inspect the first 10 rows of the duplicated trips
print("First 10 rows of duplicated trips:")
display(duplicate_ids_df.head(10))

# find number of invalid rows in accrodance with part 2 task 7, less than three GPS points
display(f"The dtype of polyline is {type(duplicate_ids_df["POLYLINE"].iloc[0])}") 
duplicate_ids_df["POLYLINE"] = duplicate_ids_df["POLYLINE"].map(json.loads) 
invalid_duplicate_ids_df = duplicate_ids_df[duplicate_ids_df["POLYLINE"].str.len() < 3]
display(invalid_duplicate_ids_df)

# check how many duplicates there are when removing invalid trips
valid_ids_df = df.drop(invalid_duplicate_ids_df.index)
print("Number of unique TRIP_IDs:", len(df["TRIP_ID"].unique()))
print("Number of rows in the new dataset:", len(valid_ids_df["TRIP_ID"]))
print("Number of duplicated TRIP_IDs:",  len(valid_ids_df["TRIP_ID"]) - len(df["TRIP_ID"].unique()))

Number of unique TRIP_IDs: 1710589
Number of rows in the dataset: 1710670
Number of duplicated TRIP_IDs: 81

Number of duplicated rows for each TRIP_ID and inspection of the trip with the highest count:


TRIP_ID
1397172149620000454    3
1372702836620000080    2
1373025987620000601    2
1397199117620000513    2
1397121307620000196    2
                      ..
1387206043620000080    2
1386947218620000080    2
1386942349620000080    2
1386167379620000080    2
1404093316620000307    2
Name: count, Length: 80, dtype: int64

,TRIP_ID,CALL_TYPE,ORIGIN_CALL,ORIGIN_STAND,TAXI_ID,TIMESTAMP,DAY_TYPE,MISSING_DATA,POLYLINE
1304113,1397172149620000454,A,30929.0,NaN,20000454,1397172149,A,False,"[[-8.618391,41.162067],[-8.6184,41.161446],[-8..."
1304114,1397172149620000454,C,NaN,NaN,20000454,1397172149,A,False,"[[-8.618391,41.162067],[-8.6184,41.161446],[-8..."
1304115,1397172149620000454,C,NaN,NaN,20000454,1397172149,A,False,"[[-8.618391,41.162067],[-8.6184,41.161446],[-8..."


First 10 rows of duplicated trips:


,TRIP_ID,CALL_TYPE,ORIGIN_CALL,ORIGIN_STAND,TAXI_ID,TIMESTAMP,DAY_TYPE,MISSING_DATA,POLYLINE
3519,1372702836620000080,B,NaN,16.0,20000080,1372702836,A,False,"[[-8.635275,41.193729],[-8.634519,41.192901]]"
3720,1372702836620000080,C,NaN,NaN,20000080,1372702836,A,False,"[[-8.62776,41.17554],[-8.62776,41.17554],[-8.6..."
21282,1373025987620000601,A,2002.0,NaN,20000601,1373025987,A,False,[]
21385,1373025987620000601,C,NaN,NaN,20000601,1373025987,A,False,"[[-8.604108,41.182569],[-8.604072,41.182605],[..."
33228,1373210896620000598,B,NaN,57.0,20000598,1373210896,A,False,"[[-8.610885,41.145525]]"
33350,1373210896620000598,C,NaN,NaN,20000598,1373210896,A,False,"[[-8.610813,41.14548],[-8.610732,41.145579],[-..."
51623,1373551710620000014,B,NaN,49.0,20000014,1373551710,A,False,"[[-8.587278,41.147325],[-8.587278,41.147325]]"
51966,1373551710620000014,C,NaN,NaN,20000014,1373551710,A,False,"[[-8.602938,41.179716],[-8.603937,41.179932],[..."
76527,1374014097620000337,C,NaN,NaN,20000337,1374014097,A,False,"[[-8.648451,41.164785]]"
76596,1374014097620000337,C,NaN,NaN,20000337,1374014097,A,False,"[[-8.648478,41.164758],[-8.648478,41.164758],[..."


"The dtype of polyline is <class 'str'>"

,TRIP_ID,CALL_TYPE,ORIGIN_CALL,ORIGIN_STAND,TAXI_ID,TIMESTAMP,DAY_TYPE,MISSING_DATA,POLYLINE
3519,1372702836620000080,B,NaN,16.0,20000080,1372702836,A,False,"[[-8.635275, 41.193729], [-8.634519, 41.192901]]"
21282,1373025987620000601,A,2002.0,NaN,20000601,1373025987,A,False,[]
33228,1373210896620000598,B,NaN,57.0,20000598,1373210896,A,False,"[[-8.610885, 41.145525]]"
51623,1373551710620000014,B,NaN,49.0,20000014,1373551710,A,False,"[[-8.587278, 41.147325], [-8.587278, 41.147325]]"
76527,1374014097620000337,C,NaN,NaN,20000337,1374014097,A,False,"[[-8.648451, 41.164785]]"
...,...,...,...,...,...,...,...,...,...
1647671,1403115272620000066,C,NaN,NaN,20000066,1403115272,A,False,"[[-8.554752, 41.172975]]"
1665292,1403407180620000242,C,NaN,NaN,20000242,1403407180,A,False,"[[-8.610687, 41.14773], [-8.611722, 41.147856]]"
1665302,1403407180620000242,C,NaN,NaN,20000242,1403407180,A,False,[]
1705646,1404093316620000307,B,NaN,57.0,20000307,1404093316,A,False,"[[-8.610831, 41.145741]]"


Number of unique TRIP_IDs: 1710589
Number of rows in the new dataset: 1710564
Number of duplicated TRIP_IDs: -25


### Conclusion:
It appears that 81 trips have does not have unique TRIP_IDs and that the rows with similar TRIP_ID are not simply duplicates of one another. It appears that for most cases there are two duplicate rows, but there is one case where there are three duplicate rows. For the duplicated rows TAXI_ID and timestamp appear to be the same. This could indicate that the first initiaton of saving the trip failed and therefore the the second or third is an attempt to correct the error. For example some of the first TRIP_IDs initiated have were few coordinates in the POLYLINE field even though MISSING_DATA is false. Since we cannot verify whether our hypothesis is correct, we have decisded to remove the IDs with multiple rows. 

It also became clear that POLYLINE was read as a str. When converting it to a list, we were able to check how many of the ids that appeared multiple times in the dataset had fewer than three GPS points. After removing the ids that appeared multiple times with less than thre GPS points, we ended up with 25 fewer ids.

In [43]:
# keep last of duplicate TRIP_IDs
df = df.drop_duplicates(subset=['TRIP_ID'], keep='last')

#### CALL_TYPE, ORIGIN_CALL and ORIGIN_STAND inspection

In [44]:
# check that there are in fact only A, B and C values for calltype
display(df["CALL_TYPE"].unique())

# function to check inconsistency in CALL_TYPE, ORIGIN_CALL and ORIGIN_STAND
def calltype_validation(column_name, call_type):
    invalid_rows = df[
                            ((df[column_name].isna()) & (df["CALL_TYPE"]==call_type)) | 
                            ((df[column_name].notna()) & (df["CALL_TYPE"]!=call_type))
                    ]
    invalid_rows_na = df[((df[column_name].isna()) & (df["CALL_TYPE"]==call_type))]
    invalid_rows_not_na = df[((df[column_name].notna()) & (df["CALL_TYPE"]!=call_type))]
    display(invalid_rows) 
    if invalid_rows.shape[0] != 0:
        print(f"Number of rows where CALL_TYPE={call_type} and {column_name} is NaN: {invalid_rows_na.shape[0]}") 
        print(f"Number of rows where CALL_TYPE!={call_type} and {column_name} is not NaN: {invalid_rows_not_na.shape[0]}") 
        return
    print(f"No inconsistency between CALL_TYPE and {column_name}")
    
# ORIGIN_CALL
calltype_validation("ORIGIN_CALL", "A")

# ORIGIN_STAND
calltype_validation("ORIGIN_STAND", "B")
# check that 

array(['C', 'B', 'A'], dtype=object)

,TRIP_ID,CALL_TYPE,ORIGIN_CALL,ORIGIN_STAND,TAXI_ID,TIMESTAMP,DAY_TYPE,MISSING_DATA,POLYLINE


No inconsistency between CALL_TYPE and ORIGIN_CALL


,TRIP_ID,CALL_TYPE,ORIGIN_CALL,ORIGIN_STAND,TAXI_ID,TIMESTAMP,DAY_TYPE,MISSING_DATA,POLYLINE
136,1372638560620000308,B,NaN,NaN,20000308,1372638560,A,False,"[[-8.58564,41.148567],[-8.585667,41.148873],[-..."
280,1372650696620000320,B,NaN,NaN,20000320,1372650696,A,False,"[[-8.6058,41.153517],[-8.606133,41.153472],[-8..."
433,1372656129620000307,B,NaN,NaN,20000307,1372656129,A,False,"[[-8.63064,41.154714],[-8.630685,41.154102],[-..."
564,1372665032620000004,B,NaN,NaN,20000004,1372665032,A,False,"[[-8.591319,41.156181],[-8.591463,41.156217],[..."
640,1372663170620000391,B,NaN,NaN,20000391,1372663170,A,False,"[[-8.630712,41.154966],[-8.630775,41.154867],[..."
...,...,...,...,...,...,...,...,...,...
1710248,1404167874620000600,B,NaN,NaN,20000600,1404167874,A,False,"[[-8.617545,41.14611],[-8.617563,41.146083],[-..."
1710472,1404151531620000001,B,NaN,NaN,20000001,1404151531,A,False,"[[-8.605638,41.153589],[-8.607051,41.153589],[..."
1710480,1404172241620000675,B,NaN,NaN,20000675,1404172241,A,False,"[[-8.617662,41.146083],[-8.617626,41.145984],[..."
1710639,1404155788620000591,B,NaN,NaN,20000591,1404155788,A,False,"[[-8.585586,41.14863],[-8.585694,41.148765],[-..."


Number of rows where CALL_TYPE=B and ORIGIN_STAND is NaN: 11302
Number of rows where CALL_TYPE!=B and ORIGIN_STAND is not NaN: 0


### Conclusion

CALL_TYPE only contains the values A, B, C as expected. 

There are no inconsitencies between CALL_TYPE and ORIGIN_CALL as all ORIGIN_CALL is only NaN when CALLTYPE is not A and it is never NaN when CALL_TYPE is A.
There are 11302 rows where CALL_TYPE is B and ORIGIN_STAND is not NaN and 0 rows where where CALL_TYPE!=B and ORIGIN_STAND is not NaN. This is not necessarily a problem and could simply be due to lacking information about start. This could be interpolated by analysing the start from the POLYLINE column.

#### Unique TAXI_ID inspection - Check that the same taxi cannot do two trips at once

In [45]:
# Convert POLYLINE to list
df["POLYLINE"] = df["POLYLINE"].map(json.loads) 

KeyboardInterrupt: 

In [46]:
# Calculate the end of each trip based on POLYLINE
df["LENGTH"] = 15*(df["POLYLINE"].str.len() - 1).clip(lower=0) # clip to avoid -15 seconds and subtract 1 since the first geopoint is sampled after 0 seconds
df["TIMESTAMP_END"] = df["TIMESTAMP"] + df["LENGTH"]

# sort values by TAXI_ID and TIMESTAMP and check for overlapping trips done by the same taxi
porto_df_by_taxi = df.sort_values(by=["TAXI_ID", "TIMESTAMP"])
overlap = (porto_df_by_taxi["TAXI_ID"] == porto_df_by_taxi["TAXI_ID"].shift(1)) & (porto_df_by_taxi["TIMESTAMP_END"].shift(1) > porto_df_by_taxi["TIMESTAMP"])
porto_df_by_taxi["OVERLAP"] = overlap | overlap.shift(-1, fill_value=False)

# for inspection of result
display(porto_df_by_taxi[porto_df_by_taxi["OVERLAP"]].sort_values(by = ["TAXI_ID", "TIMESTAMP"]))

/var/folders/z0/cqhkbsv975qb9mj7pp6vf0v40000gn/T/ipykernel_60218/3144091698.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["LENGTH"] = 15*(df["POLYLINE"].str.len() - 1).clip(lower=0) # clip to avoid -15 seconds and subtract 1 since the first geopoint is sampled after 0 seconds
/var/folders/z0/cqhkbsv975qb9mj7pp6vf0v40000gn/T/ipykernel_60218/3144091698.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["TIMESTAMP_END"] = df["TIMESTAMP"] + df["LENGTH"]


,TRIP_ID,CALL_TYPE,ORIGIN_CALL,ORIGIN_STAND,TAXI_ID,TIMESTAMP,DAY_TYPE,MISSING_DATA,POLYLINE,LENGTH,TIMESTAMP_END,OVERLAP
624,1372662403620000001,B,NaN,28.0,20000001,1372662403,A,False,"[[-8.584353,41.163174],[-8.585289,41.162994],[...",6555,1372668958,True
853,1372666377620000001,A,2002.0,NaN,20000001,1372666377,A,False,"[[-8.608824,41.153436],[-8.608815,41.153463],[...",8145,1372674522,True
1170,1372669154620000001,B,NaN,63.0,20000001,1372669154,A,False,"[[-8.609562,41.160249],[-8.609652,41.160375],[...",11730,1372680884,True
1513,1372672248620000001,A,2002.0,NaN,20000001,1372672248,A,False,"[[-8.608869,41.15349],[-8.608851,41.153508],[-...",12795,1372685043,True
1854,1372676157620000001,B,NaN,10.0,20000001,1372676157,A,False,"[[-8.607096,41.150331],[-8.607096,41.150313],[...",16920,1372693077,True
...,...,...,...,...,...,...,...,...,...,...,...,...
1575497,1401893718620000969,C,NaN,NaN,20000969,1401893718,A,False,"[[-8.644887,41.15898]]",315,1401894033,True
1575499,1401901637620000969,C,NaN,NaN,20000969,1401901637,A,False,"[[-8.645049,41.158998]]",330,1401901967,True
1580440,1401901663620000969,C,NaN,NaN,20000969,1401901663,A,False,"[[-8.645049,41.159016]]",330,1401901993,True
74733,1373983173620000980,C,NaN,NaN,20000980,1373983173,A,False,"[[-8.690139,41.240133]]",330,1373983503,True


### Conclusion
There appear to be 1714 rows in which the same TAXI_ID has some overlap between two different trips. There appears to only be some seconds difference. We therefore assumed that this is due to clocking errors and not actual anomalies in the data.

### TIMESTAMP inspection

In [47]:
# Start time of the trip, expressed as Unix time (seconds since 1970-01-01). Should be converted to a standard DATETIME format for queries.
df["DATE_AND_HOUR"] = pd.to_datetime(df["TIMESTAMP"], unit="s")
min_time = df["DATE_AND_HOUR"].min()
max_time = df["DATE_AND_HOUR"].max()
display(df["DATE_AND_HOUR"])
print(f"The first timestamp is at {min_time} whereas the last timestamp is at {max_time}")

/var/folders/z0/cqhkbsv975qb9mj7pp6vf0v40000gn/T/ipykernel_60218/1753029146.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["DATE_AND_HOUR"] = pd.to_datetime(df["TIMESTAMP"], unit="s")


0         2013-07-01 00:00:58
1         2013-07-01 00:08:23
2         2013-07-01 00:02:31
3         2013-07-01 00:00:54
4         2013-07-01 00:04:51
                  ...        
1710665   2014-06-30 23:37:43
1710666   2014-06-30 23:36:07
1710667   2014-01-03 10:41:56
1710668   2014-06-30 15:23:46
1710669   2014-06-30 19:39:07
Name: DATE_AND_HOUR, Length: 1710589, dtype: datetime64[ns]

The first timestamp is at 2013-07-01 00:00:53 whereas the last timestamp is at 2014-06-30 23:59:56


### DAYTIME inspection

In [48]:
# Indicates whether the trip occurred on a normal day, a holiday, or the day before a holiday. A: Normal day (workday or weekend), B: Holiday or special day, C: Day before a holiday.
display(df["DAY_TYPE"].unique())

array(['A'], dtype=object)

### MISSING_DATA inspection

In [49]:
# Boolean flag indicating whether there are missing GPS points in the POLYLINE. FALSE means the data is complete, TRUE means some points are missing.
display(df["MISSING_DATA"].unique())
missing_data = df[df["MISSING_DATA"]]

array([False,  True])

In [50]:
df_explode_popyline = missing_data[["TRIP_ID", "POLYLINE"]].explode("POLYLINE").copy()

df_explode_popyline[["Longitude", "Latitude"]] = pd.DataFrame(
    df_explode_popyline["POLYLINE"].tolist(),
    index=df_explode_popyline.index
)

df_explode_popyline = df_explode_popyline.drop(columns="POLYLINE").reset_index(drop=True)

display(df_explode_popyline)

ValueError: Columns must be same length as key

In [ ]:
# Remove trips with Polyline lenght less than 1. 
# unique trip_ids with missing polyline points
ids_missing_data = missing_data["TRIP_ID"].unique()



import plotly.graph_objects as go

fig = go.Figure()

for trip_id, trip in df_explode_popyline.groupby("TRIP_ID"):
    fig.add_trace(
        go.Scattermap(
            lat=trip["Latitude"],
            lon=trip["Longitude"],
            mode="lines",
            name=str(trip_id),
            hovertemplate=(
                f"Trip: {trip_id}<br>"
                "Lat: %{lat}<br>"
                "Lon: %{lon}"
                "<extra></extra>"
            )
        )
    )

fig.update_layout(
    map=dict(
        style="open-street-map",
        zoom=12,
        center={
            "lat": df_explode_popyline["Latitude"].mean(),
            "lon": df_explode_popyline["Longitude"].mean()
        }
    ),
    height=700,
    margin=dict(l=0, r=0, t=40, b=0),
    title="Trips"
)

fig.show()

In [ ]:
# POLYLINE: A list of GPS points forming the trajectory. Each point is represented as [longitude, latitude], sampled at 15-second intervals.
df["POLYLINE"].unique()

TypeError: unhashable type: 'list'

# Cleaning data

In [ ]:
# keep last of the duplicate TRIP_IDs
df = df.drop_duplicates(subset=['TRIP_ID'], keep='last')

# interpolate point B based on trips that have been done within a close distanse

# convert to datetime

# write to csv